# ZemaVoice Stage 7 — Movie Audio Sync Studio (personal prototype)
Upload a video MP4 and provide a CSV timeline with Amharic dialogue. Generate each line using the **existing single-voice** Amharic baseline, place it at the requested start time, mix with the video's original audio, and export an MP4 plus separate WAV clips and a report.

**Not yet included:** independent female/children's voices or automatic mouth/lip-sync. The `voice` column is a character label only. Overlapping dialogue is mixed, not assigned different speakers. Lines that extend past the next line or video end are flagged, not silently sped up.

**License:** Meta `facebook/mms-tts-amh` is CC BY-NC 4.0, for noncommercial evaluation only. Use only with video and audio you have rights to edit.

In [ ]:
!pip -q install 'transformers>=4.44,<5' 'accelerate>=0.26' 'soundfile>=0.12' 'uroman>=1.3'
!ffmpeg -version | head -n 1


In [ ]:
import os, csv, json, subprocess, shutil, zipfile
from pathlib import Path
import numpy as np, torch, soundfile as sf
from IPython.display import Audio, display
from google.colab import files
from transformers import VitsModel,VitsTokenizer,set_seed
from uroman import Uroman
from scipy.signal import resample_poly
from math import gcd
root=Path('/content/zemavoice_stage7');root.mkdir(exist_ok=True)
model_id='facebook/mms-tts-amh'
tokenizer=VitsTokenizer.from_pretrained(model_id)
model=VitsModel.from_pretrained(model_id).eval()
device='cuda' if torch.cuda.is_available() else 'cpu'
model=model.to(device)
romanizer=Uroman()
sr=int(model.config.sampling_rate)
print('Ready:',device,'| Speech sample rate:',sr)


## 1. Upload your MP4
Use a short 15–60 second MP4 for the first test. This prototype creates a temporary workspace in Colab.

In [ ]:
uploaded=files.upload()
videos=[name for name in uploaded if name.lower().endswith(('.mp4','.mov','.mkv','.webm'))]
assert len(videos)==1,'Please upload exactly one video file in this cell.'
video_path=Path('/content')/videos[0]
probe=subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(video_path)],capture_output=True,text=True,check=True)
video_seconds=float(probe.stdout.strip())
print('Video:',video_path.name,'| Duration:',round(video_seconds,2),'seconds')


## 2. Edit your Amharic dialogue timeline
Each row has a `start` in seconds, a `character` label, and `text`. Set `keep_original_audio` to `True` if you want to retain the video's existing soundtrack. `original_volume` is 0–1. The character labels do **not** change the voice yet.

Example: at 2.0 seconds narrator speaks; at 9.0 seconds a character speaks.

In [ ]:
timeline=[
  {'start':2.0,'character':'Narrator','text':'በአንድ ጨለማ ሌሊት፣ መንደሩ በጸጥታ ተውጦ ነበር።'},
  {'start':9.0,'character':'Character 1','text':'ወደ ሰማይ ተመልከቱ።'},
]
keep_original_audio=True
original_volume=0.20  # 0=mute, 1=original level
voice_volume=1.0
set_seed(42)
assert 0<=original_volume<=1 and 0<=voice_volume<=2
assert 1<=len(timeline)<=100,'Use 1–100 lines per run.'
assert all(0<=float(x['start'])<video_seconds for x in timeline),'Each start time must be within the video.'
assert all(x['text'].strip() for x in timeline),'Empty dialogue text found.'
timeline=sorted(timeline,key=lambda x:float(x['start']))
print('Ready to generate',len(timeline),'dialogue lines')


## 3. Generate dialogue and mix at exact start times
This aligns the **start** of each spoken line. It does not change facial animation. Long lines may overlap; check the warnings and edit start times or text.

In [ ]:
clips=root/'clips';clips.mkdir(exist_ok=True)
total_samples=int(np.ceil(video_seconds*sr))
mixed=np.zeros(total_samples,dtype=np.float32)
report=[]
for i,line in enumerate(timeline):
    romanized=romanizer.romanize_string(line['text'])
    assert romanized,f'Romanization failed at line {i+1}'
    inputs=tokenizer(romanized,return_tensors='pt').to(device)
    with torch.inference_mode():
        wav=model(**inputs).waveform[0].detach().cpu().float().numpy()
    start=float(line['start']);offset=round(start*sr)
    end=min(total_samples,offset+len(wav))
    if end>offset:mixed[offset:end]+=wav[:end-offset]*voice_volume
    name=f'line_{i+1:03d}.wav'
    sf.write(clips/name,wav,sr)
    duration=len(wav)/sr
    next_start=float(timeline[i+1]['start']) if i+1<len(timeline) else video_seconds
    warnings=[]
    if start+duration>next_start:warnings.append('Line extends beyond next start or video end')
    report.append({'line':i+1,'start':start,'end':round(start+duration,3),'duration':round(duration,3),'character':line['character'],'text':line['text'],'clip':name,'warnings':warnings})
    print(f"Line {i+1}: {line['character']} | {start:.2f}–{start+duration:.2f}s",'WARNING' if warnings else '')
mixed=np.clip(mixed,-0.98,0.98)
voice_path=root/'aligned_dialogue.wav';sf.write(voice_path,mixed,sr)
(root/'timing_report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
display(Audio(mixed,rate=sr))


In [ ]:
output=root/'zemavoice_synced_movie.mp4'
if keep_original_audio:
    cmd=['ffmpeg','-y','-i',str(video_path),'-i',str(voice_path),'-filter_complex',f'[0:a]volume={original_volume}[bg];[1:a]volume=1[voice];[bg][voice]amix=inputs=2:duration=first:dropout_transition=0[a]','-map','0:v:0','-map','[a]','-c:v','copy','-c:a','aac','-b:a','192k','-t',str(video_seconds),str(output)]
    result=subprocess.run(cmd,capture_output=True,text=True)
    if result.returncode!=0 and 'matches no streams' in result.stderr:
        print('No original audio stream detected. Exporting voice only.')
        keep_original_audio=False
    elif result.returncode!=0:raise RuntimeError(result.stderr[-2500:])
if not keep_original_audio:
    subprocess.run(['ffmpeg','-y','-i',str(video_path),'-i',str(voice_path),'-map','0:v:0','-map','1:a:0','-c:v','copy','-c:a','aac','-b:a','192k','-t',str(video_seconds),str(output)],check=True,capture_output=True,text=True)
print('Created:',output,'| Size:',round(output.stat().st_size/1e6,1),'MB')


In [ ]:
archive=Path('/content/zemavoice_stage7_export.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(root.rglob('*')):
        if p.is_file():z.write(p,p.relative_to(root))
print('ZIP contains synced MP4, aligned dialogue WAV, individual clips and timing report.')
files.download(str(archive))
